# 1. TIỀN XỬ LÝ DỮ LIỆU (Data Preprocessing)

**Bài toán:** Phân khúc khách hàng & dự đoán phản hồi chiến dịch Marketing  
**Dữ liệu:** `marketing_campaign.csv` (2240 mẫu × 29 thuộc tính)  
**Biến mục tiêu:** `Response` (0/1 – chấp nhận hay từ chối chiến dịch)

---

## Nội dung
1. [1.1 Đọc dữ liệu và mô tả cấu trúc](#1.1)
2. [1.2 Thống kê sơ bộ](#1.2)
3. [1.3 Trực quan hóa dữ liệu thô](#1.3)
4. [1.4 Xử lý dữ liệu lỗi (missing, outlier, bất hợp lệ)](#1.4)
5. [1.5 Kỹ thuật tạo đặc trưng (Feature Engineering)](#1.5)
6. [1.6 Chuyển đổi dữ liệu (One-Hot Encoding, Chuẩn hóa)](#1.6)
7. [1.7 Mô tả dữ liệu sau chuẩn hóa](#1.7)
8. [1.8 Lưu dữ liệu đã xử lý](#1.8)

In [ ]:
# === Import thư viện ===
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc project vào path để import src/
sys.path.insert(0, os.path.abspath('..'))

from src import config
from src import preprocessing

# Cấu hình hiển thị
pd.set_option('display.max_columns', 35)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = config.FIGURE_DPI

print(f'Random Seed: {config.RANDOM_SEED}')
print(f'Project Root: {config.PROJECT_ROOT}')

<a id='1.1'></a>
## 1.1 Đọc dữ liệu và mô tả cấu trúc

Dữ liệu gốc là file `marketing_campaign.csv` sử dụng dấu phân cách **Tab** (`\t`).

In [ ]:
# Đọc dữ liệu thô
df_raw = preprocessing.load_raw()

print(f'Kích thước tập dữ liệu: {df_raw.shape[0]} mẫu × {df_raw.shape[1]} thuộc tính')
print(f'Bộ nhớ sử dụng: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Xem 5 dòng đầu tiên
df_raw.head()

In [ ]:
# Thông tin cấu trúc dữ liệu: kiểu dữ liệu, số lượng non-null
df_raw.info()

In [ ]:
# Mô tả ý nghĩa các trường dữ liệu
feature_descriptions = pd.DataFrame({
    'Tên cột': df_raw.columns,
    'Kiểu dữ liệu': df_raw.dtypes.values,
    'Non-Null': df_raw.notnull().sum().values,
    'Null': df_raw.isnull().sum().values,
    'Unique': df_raw.nunique().values,
    'Mô tả': [
        'Mã định danh khách hàng',
        'Năm sinh',
        'Trình độ học vấn',
        'Tình trạng hôn nhân',
        'Thu nhập hộ gia đình hàng năm',
        'Số trẻ nhỏ trong nhà',
        'Số thiếu niên trong nhà',
        'Ngày đăng ký làm khách hàng',
        'Số ngày kể từ lần mua hàng gần nhất',
        'Chi tiêu cho rượu vang (2 năm gần nhất)',
        'Chi tiêu cho trái cây',
        'Chi tiêu cho thịt',
        'Chi tiêu cho cá',
        'Chi tiêu cho đồ ngọt',
        'Chi tiêu cho vàng/trang sức',
        'Số lần mua qua khuyến mãi',
        'Số lần mua qua website',
        'Số lần mua qua catalog',
        'Số lần mua trực tiếp tại cửa hàng',
        'Số lần truy cập web/tháng',
        'Chấp nhận chiến dịch 3 (0/1)',
        'Chấp nhận chiến dịch 4 (0/1)',
        'Chấp nhận chiến dịch 5 (0/1)',
        'Chấp nhận chiến dịch 1 (0/1)',
        'Chấp nhận chiến dịch 2 (0/1)',
        'Đã phàn nàn trong 2 năm gần nhất (0/1)',
        'Chi phí liên hệ (hằng số = 3)',
        'Doanh thu (hằng số = 11)',
        'Phản hồi chiến dịch cuối (TARGET: 0/1)',
    ]
})
feature_descriptions

<a id='1.2'></a>
## 1.2 Thống kê sơ bộ

In [ ]:
# Thống kê mô tả cho các biến số
df_raw.describe().T

In [ ]:
# Thống kê cho các biến phân loại
print('=== Education ===')
print(df_raw['Education'].value_counts())
print(f'\n=== Marital_Status ===')
print(df_raw['Marital_Status'].value_counts())
print(f'\n=== Dt_Customer (phạm vi) ===')
print(f"Min: {df_raw['Dt_Customer'].min()}, Max: {df_raw['Dt_Customer'].max()}")

In [ ]:
# Kiểm tra giá trị thiếu (Missing Values)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Số lượng null': missing, 'Tỷ lệ (%)': missing_pct})
missing_df = missing_df[missing_df['Số lượng null'] > 0]
print('Các cột có giá trị thiếu:')
print(missing_df if len(missing_df) > 0 else 'Không có cột nào bị thiếu giá trị.')

In [ ]:
# Kiểm tra bản ghi trùng lặp
n_duplicates = df_raw.duplicated().sum()
print(f'Số bản ghi trùng lặp: {n_duplicates}')

In [ ]:
# Phân phối biến mục tiêu (Response)
response_counts = df_raw['Response'].value_counts()
response_pct = df_raw['Response'].value_counts(normalize=True) * 100
print('Phân phối biến mục tiêu Response:')
print(pd.DataFrame({'Số lượng': response_counts, 'Tỷ lệ (%)': response_pct.round(2)}))
print(f'\n=> Dữ liệu KHÔNG CÂN BẰNG: lớp 1 chiếm chỉ {response_pct[1]:.1f}%')

<a id='1.3'></a>
## 1.3 Trực quan hóa dữ liệu thô

In [ ]:
# --- 1.3.1 Phân phối biến mục tiêu ---
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#3498db', '#e74c3c']
df_raw['Response'].value_counts().plot(kind='bar', color=colors, edgecolor='black', ax=ax)
ax.set_title('Phân phối biến mục tiêu (Response)', fontsize=14, fontweight='bold')
ax.set_xlabel('Response')
ax.set_ylabel('Số lượng')
ax.set_xticklabels(['0 - Từ chối', '1 - Chấp nhận'], rotation=0)
for i, v in enumerate(df_raw['Response'].value_counts().values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'target_distribution.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1.3.2 Biểu đồ Missing Values ---
fig, ax = plt.subplots(figsize=(12, 5))
null_counts = df_raw.isnull().sum().sort_values(ascending=False)
null_counts = null_counts[null_counts > 0]
if len(null_counts) > 0:
    null_counts.plot(kind='barh', color='coral', edgecolor='black', ax=ax)
    ax.set_title('Số lượng giá trị thiếu (Missing Values) theo cột', fontsize=14, fontweight='bold')
    ax.set_xlabel('Số giá trị null')
    for i, v in enumerate(null_counts.values):
        ax.text(v + 0.3, i, f'{v} ({v/len(df_raw)*100:.1f}%)', va='center')
else:
    ax.text(0.5, 0.5, 'Không có giá trị thiếu', ha='center', va='center', fontsize=14, transform=ax.transAxes)
plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'missing_values.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1.3.3 Phân phối các biến số chính (trước xử lý) ---
num_cols_to_plot = ['Year_Birth', 'Income', 'Recency',
                    'MntWines', 'MntMeatProducts', 'MntFruits',
                    'NumWebPurchases', 'NumStorePurchases', 'NumCatalogPurchases']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Phân phối các biến số chính (DỮ LIỆU THÔ)', fontsize=16, fontweight='bold', y=1.02)

for idx, col in enumerate(num_cols_to_plot):
    ax = axes[idx // 3, idx % 3]
    df_raw[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(df_raw[col].mean(), color='red', linestyle='--', label=f'Mean={df_raw[col].mean():.0f}')
    ax.axvline(df_raw[col].median(), color='green', linestyle='-', label=f'Median={df_raw[col].median():.0f}')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'raw_distributions.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1.3.4 Phân phối biến phân loại ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Education
edu_order = df_raw['Education'].value_counts().index
sns.countplot(data=df_raw, x='Education', order=edu_order, hue='Response',
              palette={0: '#3498db', 1: '#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Phân phối Education theo Response', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Trình độ học vấn')
axes[0].set_ylabel('Số lượng')

# Marital Status
ms_order = df_raw['Marital_Status'].value_counts().index
sns.countplot(data=df_raw, x='Marital_Status', order=ms_order, hue='Response',
              palette={0: '#3498db', 1: '#e74c3c'}, ax=axes[1], edgecolor='black')
axes[1].set_title('Phân phối Marital_Status theo Response', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tình trạng hôn nhân')
axes[1].set_ylabel('Số lượng')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'categorical_distributions.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1.3.5 Boxplot phát hiện outlier ---
outlier_cols = ['Year_Birth', 'Income', 'MntWines', 'MntMeatProducts', 'MntSweetProducts', 'MntGoldProds']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Boxplot phát hiện ngoại lai (Outliers) - DỮ LIỆU THÔ', fontsize=16, fontweight='bold', y=1.02)

for idx, col in enumerate(outlier_cols):
    ax = axes[idx // 3, idx % 3]
    sns.boxplot(data=df_raw, y=col, ax=ax, color='lightcoral', width=0.5)
    ax.set_title(col, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'raw_boxplots.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

<a id='1.4'></a>
## 1.4 Xử lý dữ liệu lỗi (Missing, Outlier, Bất hợp lệ)

Các vấn đề phát hiện được:
1. **Giá trị thiếu:** Cột `Income` có 24 giá trị null (1.07%).
2. **Cột hằng số vô nghĩa:** `Z_CostContact` (luôn = 3) và `Z_Revenue` (luôn = 11) – không có giá trị phân biệt.
3. **Cột định danh:** `ID` – không mang thông tin cho mô hình.
4. **Outlier bất hợp lý:**
   - `Year_Birth`: Giá trị 1893, 1899, 1900 → tuổi > 110 (phi thực tế).
   - `Income`: Giá trị 666,666 → xa vời so với Q3 = 68,522.

In [ ]:
# Kiểm tra chi tiết các bản ghi ngoại lai
print('=== Outlier Year_Birth (< 1920) ===')
print(df_raw[df_raw['Year_Birth'] < 1920][['ID', 'Year_Birth', 'Income', 'Education', 'Marital_Status']])

print('\n=== Outlier Income (> 200,000) ===')
print(df_raw[df_raw['Income'] > 200000][['ID', 'Year_Birth', 'Income', 'Education', 'Marital_Status']])

print('\n=== Marital_Status bất thường ===')
abnormal_ms = df_raw[df_raw['Marital_Status'].isin(['Absurd', 'YOLO', 'Alone'])]
print(f'Số bản ghi: {len(abnormal_ms)}')
print(abnormal_ms[['ID', 'Marital_Status', 'Income', 'Education']])

In [ ]:
# Áp dụng hàm clean_raw_data từ preprocessing module
df_cleaned = preprocessing.clean_raw_data(df_raw)

print(f'Kích thước TRƯỚC khi làm sạch: {df_raw.shape}')
print(f'Kích thước SAU khi làm sạch:   {df_cleaned.shape}')
print(f'  - Số cột bị loại bỏ: {df_raw.shape[1] - df_cleaned.shape[1]} (ID, Z_CostContact, Z_Revenue)')
print(f'  - Số mẫu bị loại bỏ: {df_raw.shape[0] - df_cleaned.shape[0]} (outlier Year_Birth/Income)')
print(f'  - Giá trị null còn lại: {df_cleaned.isnull().sum().sum()}')

In [ ]:
# Xem lại phân phối Year_Birth và Income sau khi lọc outlier
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['Year_Birth'], bins=30, alpha=0.5, color='red', label='Trước', edgecolor='black')
axes[0].hist(df_cleaned['Year_Birth'], bins=30, alpha=0.7, color='steelblue', label='Sau', edgecolor='black')
axes[0].set_title('Year_Birth: Trước vs Sau lọc outlier', fontsize=13, fontweight='bold')
axes[0].legend()

axes[1].hist(df_raw['Income'].dropna(), bins=40, alpha=0.5, color='red', label='Trước', edgecolor='black')
axes[1].hist(df_cleaned['Income'].dropna(), bins=40, alpha=0.7, color='steelblue', label='Sau', edgecolor='black')
axes[1].set_title('Income: Trước vs Sau lọc outlier', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'outlier_comparison.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

<a id='1.5'></a>
## 1.5 Kỹ thuật tạo đặc trưng (Feature Engineering)

Tạo các đặc trưng mới có ý nghĩa kinh doanh từ dữ liệu gốc:

| Đặc trưng mới | Công thức | Ý nghĩa |
|---|---|---|
| `Age` | `2014 - Year_Birth` | Tuổi khách hàng |
| `Customer_Tenure_Days` | `max(Dt_Customer) - Dt_Customer` | Số ngày gắn bó |
| `Marital_Group` | Gom nhóm `Marital_Status` | Partner / Single |
| `Education_Level` | Gom nhóm `Education` | Undergraduate / Graduate / Postgraduate |
| `Total_Spent` | Tổng 6 cột Mnt* | Tổng chi tiêu |
| `Total_Purchases` | Tổng 4 cột Num*Purchases | Tổng lần mua |
| `Total_Accepted_Cmp` | Tổng AcceptedCmp1-5 | Số chiến dịch đã nhận |
| `Children` | `Kidhome + Teenhome` | Tổng trẻ em |
| `Is_Parent` | `Children > 0` | Có con hay không |
| `Family_Size` | Partner status + Children | Quy mô gia đình |

In [ ]:
# Áp dụng Feature Engineering
df_featured = preprocessing.engineer_features(df_cleaned)

print(f'Kích thước sau Feature Engineering: {df_featured.shape}')
print(f'\nCác cột mới được tạo:')
new_cols = [c for c in df_featured.columns if c not in df_cleaned.columns]
for col in new_cols:
    print(f'  + {col}: {df_featured[col].dtype}, '
          f'range=[{df_featured[col].min()}, {df_featured[col].max()}]')

print(f'\nCác cột đã loại bỏ (đã thay thế bằng đặc trưng mới):')
removed_cols = [c for c in df_cleaned.columns if c not in df_featured.columns]
for col in removed_cols:
    print(f'  - {col}')

In [ ]:
# Xem dữ liệu sau feature engineering
df_featured.head()

In [ ]:
# Thống kê mô tả sau feature engineering
df_featured.describe().T

In [ ]:
# --- Trực quan hóa các đặc trưng mới ---
new_num_cols = ['Age', 'Customer_Tenure_Days', 'Total_Spent', 'Total_Purchases',
                'Total_Accepted_Cmp', 'Children', 'Family_Size']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Phân phối các đặc trưng mới (Feature Engineering)', fontsize=16, fontweight='bold', y=1.02)

for idx, col in enumerate(new_num_cols):
    ax = axes[idx // 4, idx % 4]
    sns.histplot(data=df_featured, x=col, hue='Response', kde=True,
                 palette={0: '#3498db', 1: '#e74c3c'}, ax=ax, alpha=0.6)
    ax.set_title(col, fontsize=12, fontweight='bold')

# Ẩn ô trống
axes[1, 3].set_visible(False)

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'engineered_features.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- Phân phối biến categorical mới ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_featured, x='Education_Level', hue='Response',
              palette={0: '#3498db', 1: '#e74c3c'}, ax=axes[0], edgecolor='black',
              order=['Undergraduate', 'Graduate', 'Postgraduate'])
axes[0].set_title('Education_Level theo Response', fontsize=13, fontweight='bold')

sns.countplot(data=df_featured, x='Marital_Group', hue='Response',
              palette={0: '#3498db', 1: '#e74c3c'}, ax=axes[1], edgecolor='black')
axes[1].set_title('Marital_Group theo Response', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'categorical_engineered.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

<a id='1.6'></a>
## 1.6 Chuyển đổi dữ liệu (One-Hot Encoding & Chuẩn hóa)

Xây dựng **ColumnTransformer** với:
- **Biến số:** SimpleImputer (median) → StandardScaler
- **Biến phân loại:** SimpleImputer (most_frequent) → OneHotEncoder

In [ ]:
# Tách các loại đặc trưng
num_cols, cat_cols = preprocessing.split_feature_types(df_featured, target_col='Response')

print(f'Số đặc trưng dạng số (numeric):       {len(num_cols)}')
print(f'  → {num_cols}')
print(f'\nSố đặc trưng dạng phân loại (category): {len(cat_cols)}')
print(f'  → {cat_cols}')

In [ ]:
# Xây dựng preprocessor
preprocessor = preprocessing.build_preprocessor(num_cols, cat_cols)

# Tách features và target
X = df_featured[num_cols + cat_cols]
y = df_featured['Response']

# Fit và transform
X_transformed = preprocessor.fit_transform(X)

# Lấy tên cột sau one-hot encoding
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

# Tạo DataFrame
df_transformed = pd.DataFrame(X_transformed, columns=all_feature_names)

print(f'Kích thước trước chuyển đổi: {X.shape}')
print(f'Kích thước sau chuyển đổi:  {df_transformed.shape}')
print(f'  (Tăng từ {len(cat_cols)} cột categorical → {len(cat_feature_names)} cột one-hot)')

In [ ]:
# Xem dữ liệu sau transform
df_transformed.head()

<a id='1.7'></a>
## 1.7 Mô tả dữ liệu sau chuẩn hóa

In [ ]:
# Thống kê mô tả sau chuẩn hóa
print('Thống kê dữ liệu SAU chuẩn hóa (StandardScaler cho biến số):')
df_transformed.describe().T

In [ ]:
# Kiểm tra: mean ≈ 0 và std ≈ 1 cho các cột numeric
stats = df_transformed[num_cols].agg(['mean', 'std']).T
stats.columns = ['Mean (sau chuẩn hóa)', 'Std (sau chuẩn hóa)']
print('Kiểm tra StandardScaler: Mean ≈ 0 và Std ≈ 1')
stats

In [ ]:
# --- So sánh phân phối TRƯỚC và SAU chuẩn hóa ---
compare_cols = ['Income', 'Age', 'Total_Spent', 'Total_Purchases', 'Recency', 'MntWines']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('So sánh phân phối TRƯỚC và SAU chuẩn hóa (StandardScaler)',
             fontsize=16, fontweight='bold', y=1.02)

for idx, col in enumerate(compare_cols):
    ax = axes[idx // 3, idx % 3]

    # Trước
    ax2 = ax.twinx()
    ax.hist(df_featured[col].dropna(), bins=30, alpha=0.4, color='red', label='Trước', edgecolor='black')
    ax2.hist(df_transformed[col], bins=30, alpha=0.4, color='steelblue', label='Sau (scaled)', edgecolor='black')

    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count (trước)', color='red')
    ax2.set_ylabel('Count (sau)', color='steelblue')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'before_after_scaling.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- Ma trận tương quan (Correlation Matrix) ---
fig, ax = plt.subplots(figsize=(18, 14))

# Thêm cột Response vào để xem tương quan với target
df_corr = df_transformed.copy()
df_corr['Response'] = y.values

corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 7}, vmin=-1, vmax=1)
ax.set_title('Ma trận tương quan (Correlation Matrix) - Dữ liệu sau chuẩn hóa',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'correlation_matrix.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- Top đặc trưng tương quan mạnh nhất với Response ---
response_corr = corr_matrix['Response'].drop('Response').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_n = 15
colors = ['#e74c3c' if v > 0 else '#3498db'
          for v in corr_matrix['Response'].drop('Response').reindex(response_corr.index[:top_n])]
response_corr[:top_n].plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.set_title(f'Top {top_n} đặc trưng tương quan mạnh nhất với Response',
             fontsize=14, fontweight='bold')
ax.set_xlabel('|Correlation|')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(config.FIGURES_PREPROCESSING_DIR / 'top_correlations.png', dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.show()

print('\nTop 10 đặc trưng tương quan với Response:')
for feat, val in response_corr[:10].items():
    actual = corr_matrix.loc[feat, 'Response']
    direction = '↑ Dương' if actual > 0 else '↓ Âm'
    print(f'  {feat:30s}  r = {actual:+.4f}  ({direction})')

<a id='1.8'></a>
## 1.8 Lưu dữ liệu đã xử lý

In [ ]:
# Lưu dữ liệu đã feature engineering (trước chuẩn hóa) – dùng cho notebook 02, 03, 04, 05
path_featured = preprocessing.save_processed(df_featured, 'cleaned_marketing_campaign.csv')
print(f'Đã lưu dữ liệu đã xử lý: {path_featured}')
print(f'  Shape: {df_featured.shape}')

# Lưu dữ liệu đã chuẩn hóa (sau ColumnTransformer)
df_final = df_transformed.copy()
df_final['Response'] = y.values
path_transformed = preprocessing.save_processed(df_final, 'transformed_marketing_campaign.csv')
print(f'\nĐã lưu dữ liệu đã chuẩn hóa: {path_transformed}')
print(f'  Shape: {df_final.shape}')

---
## Tổng kết Tiền xử lý

| Bước | Chi tiết | Kết quả |
|---|---|---|
| **Đọc dữ liệu** | `marketing_campaign.csv`, tab-separated | 2240 mẫu × 29 cột |
| **Loại cột vô nghĩa** | Drop `ID`, `Z_CostContact`, `Z_Revenue` | -3 cột |
| **Lọc outlier** | `Year_Birth < 1920`, `Income > 200,000` | -4 mẫu |
| **Feature Engineering** | 10 đặc trưng mới, loại 4 cột gốc | +6 cột net |
| **Encoding** | One-Hot cho `Marital_Group`, `Education_Level` | +3 cột (2 → 5) |
| **Chuẩn hóa** | StandardScaler cho 29 biến số | Mean≈0, Std≈1 |
| **Kết quả cuối** | **2236 mẫu × 34 đặc trưng** | Sẵn sàng cho bước tiếp theo |